In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import calendar

np.random.seed(42)
random.seed(42)


## **✨ GENERAR DATASETS SINTÉTICOS**

+ Es necesario tener en cuenta dataset reales que nos ayduen a formar unos datos más reales.
+ Crearé 5 nuevos datasets, uno por año, teniendo en cuenta las columnas necesarias en un e-commerce real. Tendré en cuenta los datos reales del INE para establecer las probabilidades de destino y reserva.

**1. CARGA Y LIMPIEZA DATASETS INE TURISMO (2021 - 2025)**

In [4]:
# Carga del archivo CSV: Personas viajeras por edad, según destino y duración del viaje

ine_edad_destino = pd.read_csv(
    r"C:\Users\Dell\Desktop\BOOTCAMP_DATA\ESPECIALIZACIÓN\S13_PROYECTO\DATOS INE\Personas viajeras por edad, según destino y duración del viaje.csv",
    sep=";",
    encoding="latin-1"
)

# Limpieza y conversión

ine_edad_destino["Total"] = (
    ine_edad_destino["Total"]
    .replace("..", np.nan)
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# Probabilidad real

ine_edad_destino["prob"] = ine_edad_destino["Total"] / 100.0

# Carga del archivo Excel: Viajes por destino principal (extranjero y comunidades autónomas)

ine_destinos = pd.read_excel(r"C:\Users\Dell\Desktop\BOOTCAMP_DATA\ESPECIALIZACIÓN\S13_PROYECTO\DATOS INE\Viajes, pernoctaciones, duración media y gasto por destino principal (extranjero y comunidades autónomas).xlsx"
)

c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


**2. DISTRIBUCIÓN REAL POR COMUNIDAD (INE 2025)**

* Creamos una lista con los nombres de las comunidades autónomas españolas. La usaré para asignar a cada cliente una comunidad de residencia respetando la distribución real de viajes del INE.

*  Creamos un array de numpy con los porcentajes de viajes que el INE da para 2025 por comunidad autónoma, según los archivos cargados anteriormente.

* Hacemos una distribución de probabilidades con el array *viajes_2025*


In [7]:
comunidades = [
    "Andalucía","Aragón","Asturias","Balears","Canarias","Cantabria",
    "Castilla y León","Castilla-La Mancha","Cataluña","Comunitat Valenciana",
    "Extremadura","Galicia","Madrid","Murcia","Navarra","País Vasco","La Rioja"
]

viajes_2025 = np.array([
    16.5, 3.5, 2.3, 1.8, 3.5, 2.2,
    8.2, 5.9, 12.7, 9.7, 2.2, 4.9,
    6.9, 2.0, 1.3, 2.3, 1.0
])

dist_comunidades = viajes_2025 / viajes_2025.sum()

**3. CATÁLOGOS DE DESTINOS**

* Creamos una lista de tuplas con destinos dentro de España para generar 2 columnas.
* Añadimos los destinos más frecuentes dentro de los datos del INE y de mis propios datos reales.
* Añadimos un diccionario de proveedores reales, cadenas hoteles o DMC (Destination Management Company / Empresa de gestión de destinos).
* Creamos 2 diccionarios más: el tipo de habitación más común de pueden reservar según la cantidad e viajeros y el tipo de viaje. Es una reducción de viajes más comunes. Cuando los viajes van por libre, no se reserva en este tipo de canales.

        * Crucero → ticket alto, margen alto.
        * Viaje organizado → Egipto, Japón, circuitos.
        * Solo hotel → escapadas urbanas, last minute.
        * Camping → verano, familias, bajo coste.

In [14]:
destinos_espana = [
    ("Madrid", "España"), ("Barcelona", "España"), ("Sevilla", "España"), ("Valencia", "España"), ("Málaga", "España"),
    ("Granada", "España"), ("Benidorm", "España"), ("Santiago de Compostela", "España"), ("Bilbao", "España"), ("San Sebastián", "España"),
    ("Palma de Mallorca", "España"), ("Córdoba", "España"), ("Zaragoza", "España"), ("Salamanca", "España"), ("Toledo", "España"), 
    ("Cáceres", "España"), ("Santander", "España"), ("Gijón", "España"), ("Las Palmas de Gran Canaria", "España"), ("Alicante", "España")
]

destinos_extranjero = [
    ("Roma", "Italia"), ("Florencia", "Italia"), ("Venecia", "Italia"), ("Sicilia", "Italia"), ("Milán", "Italia"),
    ("París", "Francia"), ("Carcassone", "Francia"), ("Perpignan", "Francia"), 
    ("Londres", "Reino Unido"),
    ("Lisboa", "Portugal"), ("Oporto", "Portugal"),
    ("Praga", "República Checa"), ("Viena", "Austria"), ("Varsovia", "Polonia"),
    ("Ámsterdam", "Países Bajos"), ("Brujas", "Bélgica"),
    ("Atenas", "Grecia"),
    ("Budapest", "Hungría"),
    ("Andorra", "Andorra"),
    ("La Habana / Varadero", "Cuba"),
    ("Circuito Japón", "Japón"),
    ("Fiordos Noruegos", "Noruega"), ("Laponia", "Noruega"), 
    ("Circuito Islandia", "Islandia"),
    ("Crucero por El Nilo", "Egipto"), ("Hurghada", "Egipto"), 
    ("Marrakech", "Marruecos"), ("Ciudades Imperiales y el desierto", "Marruecos"),
    ("Circuito Jordania", "Jordania"),
    ("Nueva York", "EEUU"), ("Circuito Coste Oeste", "EEUU"),
    ("Bangkok", "Tailandia"), ("Phuket", "Tailandia"), ("Circuito Vietnam", "Vietnam"), ("Bali", "Indonesia")

]

proveedores = ["Explora Traveler","Different Roads","ON Hotels","Nomada","Bambara Travel","Viajes Olympia", "Ohtels", "R2 Hoteles", "BE Live", "Ilunion", "La Beduina", "Holiday World", "Taiko", "World2Meet", "Turismark", "Traveltino", "Pierre & Vacances", "Sidertours", "Samatravel", "Avoris", "Magic Costa"]
tipos_habitacion = ["Doble","Triple","Suite","Familiar","Individual","Bungalow"]
tipos_viaje = ["crucero","viaje organizado","solo hotel","camping"]


# asignamos el tipo de viaje a cada páis, ya que un crucero por Andorra sería un dato no realista
tipo_viaje_por_pais = {
    "Egipto": ["crucero"],
    "Italia": ["solo hotel"], "Francia": ["solo hotel"], "Reino Unido": ["solo hotel"],
    "Polonia": ["solo hotel"], "Austria": ["solo hotel"], "Grecia": ["solo hotel"],
    "Bélgica": ["solo hotel"], "Hungría": ["solo hotel"], "Portugal": ["solo hotel"],
    "España": ["solo hotel", "camping"], "Andorra": ["solo hotel", "camping"],
    "Noruega": ["viaje organizado"], "Japón": ["viaje organizado"],
    "Jordania": ["viaje organizado"], "Islandia": ["viaje organizado"],
    "Cuba": ["viaje organizado"], "EEUU": ["viaje organizado"],
    "Tailandia": ["viaje organizado"], "Vietnam": ["viaje organizado"],
    "Indonesia": ["viaje organizado"],
}

**4. CLIENTES RECURRENTES**

Necesitamos que nuestro dataset sea lo más real posible. Un datset de una plataforma ecommerce, con clientes que repiten, con historial, con segmentación, y con coherencia demográfica.

* Creamos una función que genera clientes únicos. Partimos de una **base de 5.000 clientes en 2021** y cada año se añaden clientes nuevos, mientras parte de los clientes de años anteriores vuelve a comprar. La **tasa de recurrencia** (% de clientes del año con más de 1 reserva) empieza en el 22% y fluctúa alrededor de ese valor.

        * El id del cliente tiene una misma estructura.
        * Edad, sexo y orien aleatorio.
        * Generamos un dataframe.


In [15]:
def generar_clientes(n, inicio=0):
    clientes = []
    for i in range(inicio, inicio + n):
        clientes.append({
            "id_cliente": f"CLT-{str(i+1).zfill(6)}",   # código estándar CLT- + número con ceros
            "edad": random.randint(18, 75),               # mayores de edad
            "sexo": random.choice(["M", "F"]),
            "comunidad": np.random.choice(comunidades, p=dist_comunidades)   # distribución INE
        })
    return pd.DataFrame(clientes)


**5. FUNCIONES AUXILIARES (INE + patrones negocio)**

Creamos diferentes funciónes que nos ayudarán a tener un dataset más real teniendo en cuenta los datos porcentuales del INE.

* Determinar el grupo utilizando la clasificación por edades que utiliza el INE. Es una clasificación con bastante granularidad, nos permite hacer un análisis más detallado. Creamos una función para determinar el grupo de edad según INE.

* Extraer probabilidades reales de duración del viaje según los datos reales del INE.

* Elegir duración del viaje según el INE. Por ejemplo, los jóvenes hacen más viajes cortos → más probabilidad de 1–3 noches.

* Elegir España vs extranjero según el INE.

In [16]:
def grupo_edad_ine(edad):
    if edad < 25:
        return "De 15 a 24 años"
    elif edad < 35:
        return "De 25 a 34 años"
    elif edad < 45:
        return "De 35 a 44 años"
    elif edad < 55:
        return "De 45 a 54 años"
    elif edad < 65:
        return "De 55 a 64 años"
    else:
        return "65 y más años"

def elegir_duracion_y_destino_flag(edad, year):  
    grupo = grupo_edad_ine(edad)

    sub = ine_edad_destino[
        (ine_edad_destino["Edad"] == grupo) &
        (ine_edad_destino["Periodo"] == year)
    ]

    p_1_3 = sub[sub["Duración"] == "De 1 a 3 noches"]["prob"].sum()
    p_4mas = sub[sub["Duración"] == "De 4 o más noches"]["prob"].sum()

    if np.isnan(p_1_3) or np.isnan(p_4mas) or (p_1_3 + p_4mas == 0):
        p_1_3, p_4mas = 0.5, 0.5

    if random.random() < p_1_3 / (p_1_3 + p_4mas):
        noches = random.randint(1, 3)
        duracion = "De 1 a 3 noches"
    else:
        noches = random.randint(4, 14)
        duracion = "De 4 o más noches"

    sub_dur = sub[sub["Duración"] == duracion]
    p_esp = sub_dur[sub_dur["Destino"] == "Solo España"]["prob"].sum()
    p_ext = sub_dur[sub_dur["Destino"] == "Solo extranjero"]["prob"].sum()

    if np.isnan(p_esp) or np.isnan(p_ext) or (p_esp + p_ext == 0):
        p_esp, p_ext = 0.8, 0.2

    es_espana = random.random() < (p_esp / (p_esp + p_ext))

    return noches, es_espana

def prob_egipto(fecha):
    if fecha < FECHA_CAIDA_EGIPTO:
        return PROB_EGIPTO_NORMAL
    return PROB_EGIPTO_POST_GUERRA[fecha.year]

def elegir_destino(edad, year, fecha_reserva):
    noches, es_espana = elegir_duracion_y_destino_flag(edad, year)

    noches = int(noches)

    if year in [2024, 2025] and random.random() < 0.20:
        return noches, random.choice([("Tokio","Japón"),("Kioto","Japón")])  # añadimos más probalbilidades para Japón debido al repunte del destino en lso 2 ultimso años

    if random.random() < prob_egipto(fecha_reserva):
        return noches, random.choice([("El Cairo","Egipto"),("Hurghada","Egipto")])  # añadimo más probabiidades a Egipto ya que suele ser un Best Seller

    if es_espana:
        return noches, random.choice(destinos_espana)
    else:
        return noches, random.choice(destinos_extranjero)

**5.1 PARÁMETROS DE NEGOCIO: CRECIMIENTO, ESTACIONALIDAD Y CAÍDA DE EGIPTO**

* **Ingresos objetivo por año**: se parte de unos ingresos base en 2021 y se aplica el crecimiento anualt teniendo en cuenta varios factores.
* **Estacionalidad mensual**: porcentaje del total anual que se vende cada mes. 
* **Egipto**: a partir del **23/10/2023** (guerra de Gaza) la probabilidad de que una reserva sea a Egipto se desploma. Esto hace que 2023 y 2024 crezcan menos.

In [17]:
# CRECIMIENTO
INGRESOS_2021 = 10_678_000  # € de importe total en 2021 (año base post covid), establecemos un numero que podría ser real

CRECIMIENTO = {     # crecimiento anual del importe total
    2022: 0.34,     # 2021 -> 2022, más incremento ya que acaban los confinamientos y reestriccionesm post-covid
    2023: 0.12,     # 2022 -> 2023 (Egipto cae desde el 23/10 por la guerra de Gaza), el incremento además se estabiliza 
    2024: 0.13,     # 2023 -> 2024 (Egipto sigue bajo) pero el crecimiento se estabiliza
    2025: 0.18,     # 2024 -> 2025 - crecimiento real aprox.
} 

INFLACION_TICKET = 0.03  # el precio medio sube un 3% anual debio a varios factores, como la guerra o reestablecer los precios post-covid

INGRESOS_OBJETIVO = {2021: INGRESOS_2021}
for y in range(2022, 2026):
    INGRESOS_OBJETIVO[y] = INGRESOS_OBJETIVO[y-1] * (1 + CRECIMIENTO[y])

# establecesmos una ESTACIONALIDAD según meses y % del total anual, picos altos en enero (early booking), Semana Santa o verano

ESTACIONALIDAD = {
    # 2021: recuperación post-COVID, inicio de año flojo y segundo semestre fuerte
    2021: [ 5.50, 3.20, 5.80, 6.50, 8.80, 12.50, 13.60, 12.90, 8.40, 7.60, 8.20, 7.00],
    # 2022: primavera y verano fuertes
    2022: [ 8.20, 4.10, 7.00, 7.90, 9.60, 13.20, 12.90, 11.60, 6.80, 5.90, 7.00, 5.80],
    # 2023: pico de junio muy marcado; oct-nov-dic bajan por Egipto
    2023: [ 9.50, 3.70, 7.60, 6.90, 10.30, 14.40, 12.00, 10.50, 6.40, 5.20, 6.60, 6.90],
    # 2024: perfil más plano, febrero y abril mejoran
    2024: [ 9.00, 4.30, 7.00, 8.10, 9.50, 13.20, 12.80, 11.20, 6.50, 5.40, 6.60, 6.40],
    # 2025: se estabiliza la estacionalidad
    2025: [ 9.80, 3.60, 7.40, 7.10, 10.40, 14.10, 12.20, 10.60, 6.00, 5.70, 6.90, 6.20],
}
# Normalizamos %
ESTACIONALIDAD = {y: np.array(v) / np.sum(v) for y, v in ESTACIONALIDAD.items()}

# EGIPTO (guerra de Gaza)
FECHA_CAIDA_EGIPTO = datetime(2023, 10, 23)
PROB_EGIPTO_NORMAL = 0.25
PROB_EGIPTO_POST_GUERRA = {2023: 0.04, 2024: 0.06, 2025: 0.12}   # recuperación lenta

# PRECIOS (€ por persona y noche, rangos por categoría)
PRECIO_PERSONA_NOCHE = {
    "camping":          (18, 35),
    "solo hotel":       (45, 110),
    "crucero":          (110, 190),
    "viaje organizado": (95, 200),
}

# RETENCIÓN DE CLIENTES
"""IMPORTANTE ERROR/CORRECCIÓN: antes cada año se generaba por separado. Las recompras de un cliente solo se generaban dentro del mismo año natural y, al cambiar de año, se hacía un segundo sorteo mucho más bajo.
 Esto nos daba como resultaso un dataset con una retención alta en los meses 1-11 y con una caída brusca a partir del mes 12 ya que cada año empezaba de nuevo.
 Ahora, a la hora de generear las reservas y los clientes, se tienen en cuenta los 5 años."""

# Clientes que ya existían antes de 2021 (la empresa ya existía antes) y vuelven a comprar en 2021
CLIENTES_PREVIOS = 2500

# Segmentos de cliente: (% de los clientes nuevos, probabilidad de volver a comprar tras cada reserva, no se tendrán en cuentas estos nombres para la segmentación final)
SEGMENTOS = {
    "ocasional": (0.62, 0.22),   # compra una vez y rara vez repite
    "habitual":  (0.24, 0.55),   # repite 1-2 veces
    "fiel":      (0.14, 0.98),   # núcleo fiel que sostiene la retención a largo plazo
}

# Meses hasta la siguiente reserva (si el cliente vuelve), de 1 a 24
GAP_MESES = np.arange(1, 25)
_escapadas = np.exp(-(GAP_MESES - 1) / 3)                 # recompras cercanas (escapadas, puentes)
_anual     = np.exp(-0.5 * ((GAP_MESES - 12) / 1.8) ** 2) # "las vacaciones del año que viene"
_resto     = (GAP_MESES >= 5).astype(float)               # recompras repartidas a medio plazo
P_GAP = 0.45 * _escapadas / _escapadas.sum() + 0.30 * _anual / _anual.sum() + 0.25 * _resto / _resto.sum()
P_GAP = P_GAP / P_GAP.sum()

# Valores de referencia para validar
TASA_RECURRENCIA = {2021: 0.18, 2022: 0.235, 2023: 0.21, 2024: 0.225, 2025: 0.205}
PCT_CLIENTES_YA_EXISTENTES = {2022: 0.18, 2023: 0.26, 2024: 0.30, 2025: 0.32}


**6. GENERACIÓN DE RESERVAS**

* Simulamos una fecha real de reservas: el **mes se sortea según el perfil de estacionalidad del año** y el número de reservas por mes es proporcional.
* Los precios parten de un precio por persona y noche según la categoría del viaje y del INE. Al final, los importes de cada mes se **reescalan para cuadrar exactamente con el importe objetivo** (crecimiento anual × estacionalidad mensual).
* **Retención continua**: cada mes primero se asignan los clientes que tenían programada una recompra y el resto de reservas del mes son clientes nuevos. Tras cada reserva el cliente decide (según su segmento) si volverá y cuántos meses tardará, sin importar si cambia de año.
* Usaremos los datos del INE para decidir duración del viaje según edad y si viajan a España vs extranjero.
* Añadimos duraciones estándar a ciertos paquetes, como por jemplo Egipto, que siempre suele tener una duración de 7 noches.
* Importante añadir picos de reservas reales: en el primer trimestre hay muchas reservas anticipadas para el verano, mejor precio. También ofertas last minute para los clietnes indecisos.
* Calcular un precio base estimado lo más real posible tenidendo el cuenta el gasto diario del viajero del INE.
* **DESCUENTO** La clave del éxito de las campañas flash son sus descuento que oscilan entre 10% y 25%.
* **CONSTRUIR LA RESERVA** 
* **CONVERTIR A DATAFRAME** con una fila por reservas y las columnas necesarias en un ecommerce real. (descartamos detalles innecesarios o dificil de simular como la dirección, teléfono o emial del cliente que para este análisis no serán necesarios)

In [18]:
def crear_reserva(year, month, cliente):
    
    day = random.randint(1, calendar.monthrange(year, month)[1])
    fecha_reserva = datetime(year, month, day)
    
    noches, (destino, pais) = elegir_destino(cliente["edad"], year, fecha_reserva)
    noches = int(noches)
    
    # duración estándar de los viajes empaquetados
    duraciones_fijas = {
    "Egipto": 7,
    "Cuba": 7,
    "Noruega": 7,
    "Jordania": 7,
    "Islandia": 5
    }

    if pais in duraciones_fijas:
        noches = duraciones_fijas[pais]
    elif pais == "Japón":
        noches = random.randint(10, 12)
    elif pais in ["EEUU", "Tailandia", "Vietnam", "Indonesia"]:
        noches = random.randint(7, 14)
    
    tipo_viaje = random.choice(
    tipo_viaje_por_pais.get(pais, ["solo hotel"])
    )
    
    # anticipación "early booking"
    if month in [6, 7, 8]:
        delta_salida = random.randint(7, 20)
    else:
        delta_salida = random.randint(15, 180)
    
    fecha_salida = fecha_reserva + timedelta(days=delta_salida)
    fecha_vuelta = fecha_salida + timedelta(days=noches)
    
    # viajeros y habitaciones
    adultos = random.randint(1, 3)
    niños = random.randint(0, 2)
    bebes = random.randint(0, 1)
    
    total = adultos + niños + bebes
    habitaciones = max(1, int(np.ceil(total / 2)))
    
    # precio extraido de los datos del INE
    lo, hi = PRECIO_PERSONA_NOCHE[tipo_viaje]
    
    inflacion = (1 + INFLACION_TICKET) ** (year - 2021)
    
    pax_pagan = adultos + 0.6 * niños
    
    precio_bruto = round(
    random.uniform(lo, hi) *
    inflacion *
    noches *
    pax_pagan,
    2
    )
    
    descuento = round(random.uniform(0.10, 0.25), 2)
    
    importe = round(
    precio_bruto * (1 - descuento),
    2
    )
    
    # LÍMITE MÁXIMO, para no generar reservas de 60K poco reales en este tipo de ecommerce
    MAX_IMPORTE = 6576
    
    if importe > MAX_IMPORTE:
        importe = MAX_IMPORTE
    
    precio_bruto = round(
    importe / (1 - descuento),
    2
    )
    
    comision = round(importe * 0.12, 2)
    
    return {
    "fecha_reserva": fecha_reserva.date(),
    "proveedor": random.choice(proveedores),
    "precio_bruto": precio_bruto,
    "importe": importe,
    "comision": comision,
    "descuento_real": descuento,
    "fecha_salida": fecha_salida.date(),
    "fecha_vuelta": fecha_vuelta.date(),
    "numero_noches": noches,
    "tipo_habitacion": random.choice(tipos_habitacion),
    "numero_habitaciones": habitaciones,
    "numero_viajeros_total": total,
    "adultos": adultos,
    "niños": niños,
    "bebes": bebes,
    "destino": destino,
    "pais_destino": pais,
    "categoria_viaje": tipo_viaje,
    "id_cliente": cliente["id_cliente"],
    "edad_cliente": cliente["edad"],
    "sexo_cliente": cliente["sexo"],
    "comunidad_autonoma_cliente": cliente["comunidad"],
    }

In [19]:
from collections import defaultdict

# clientes ficticios solo para estimar el ticket medio
clientes_piloto = generar_clientes(300).to_dict("records")

def estimar_ticket_medio(year, n_piloto=3000):
    pesos_mes = ESTACIONALIDAD[year]
    meses = np.random.choice(range(1, 13), size=n_piloto, p=pesos_mes)
    return np.mean([crear_reserva(year, int(m), random.choice(clientes_piloto))["importe"] for m in meses])


def ajustar_importes(df, objetivo_anual, pesos_mes, max_importe=6576, iteraciones=3):
    """Reescalado mensual: el importe de cada mes cuadra con objetivo_anual * % del mes.
    CORRECCIÓN: antes la variable `pesos` se sobrescribía con los pesos de los clientes antes de
    este paso, así que el reescalado usaba números sin sentido (importes x5 y la mitad topados en 6.576 €)."""
    mes = df["fecha_reserva"].dt.month
    for _ in range(iteraciones):   # se repite porque el tope máximo resta algo de importe
        for m in range(1, 13):
            mask = mes == m
            factor = (objetivo_anual * pesos_mes[m-1]) / df.loc[mask, "importe"].sum()
            df.loc[mask, "precio_bruto"] = (df.loc[mask, "precio_bruto"] * factor).round(2)
        df["importe"] = (df["precio_bruto"] * (1 - df["descuento_real"])).round(2)
        tope = df["importe"] > max_importe
        df.loc[tope, "importe"] = max_importe
        df.loc[tope, "precio_bruto"] = (max_importe / (1 - df.loc[tope, "descuento_real"])).round(2)
    df["comision"] = (df["importe"] * 0.12).round(2)
    return df


def generar_todos_los_años(años=range(2021, 2026), seed=42):
    np.random.seed(seed)
    random.seed(seed)

    clientes = {}                 # id_cliente -> datos del cliente (incluye segmento)
    agenda = defaultdict(list)    # (año, mes) -> clientes con una recompra programada ese mes
    siguiente_id = 0

    def alta_clientes(n):
        nonlocal siguiente_id
        if n <= 0:
            return []
        nuevos = generar_clientes(n, inicio=siguiente_id)
        siguiente_id += n
        nuevos["segmento"] = np.random.choice(list(SEGMENTOS), size=n, p=[v[0] for v in SEGMENTOS.values()])
        for c in nuevos.to_dict("records"):
            clientes[c["id_cliente"]] = c
        return list(nuevos["id_cliente"])

    def programar_recompra(id_cliente, year, month):
        prob_volver = SEGMENTOS[clientes[id_cliente]["segmento"]][1]
        if random.random() < prob_volver:
            gap = int(np.random.choice(GAP_MESES, p=P_GAP))
            idx = year * 12 + (month - 1) + gap
            agenda[(idx // 12, idx % 12 + 1)].append(id_cliente)   # puede caer en el año siguiente

    # clientes anteriores a 2021 que vuelven a comprar durante 2021
    for id_c in alta_clientes(CLIENTES_PREVIOS):
        m = int(np.random.choice(range(1, 13), p=ESTACIONALIDAD[años[0]]))
        agenda[(años[0], m)].append(id_c)

    dfs = []
    pendientes = []   # recompras que no caben en un mes pasan al siguiente
    for year in años:
        objetivo_anual = INGRESOS_OBJETIVO[year]
        pesos_mes = ESTACIONALIDAD[year]

        # nº de reservas del año = ingresos objetivo / ticket medio esperado, repartidas por estacionalidad
        n_reservas = int(round(objetivo_anual / estimar_ticket_medio(year)))
        reservas_mes = np.round(n_reservas * pesos_mes).astype(int)

        rows = []
        for m in range(1, 13):
            cola = pendientes + agenda.pop((year, m), [])
            random.shuffle(cola)
            huecos = reservas_mes[m-1]
            vuelven, pendientes = cola[:huecos], cola[huecos:]
            nuevos = alta_clientes(huecos - len(vuelven))   # el resto del mes son clientes nuevos
            for id_c in vuelven + nuevos:
                rows.append(crear_reserva(year, m, clientes[id_c]))
                programar_recompra(id_c, year, m)

        df = pd.DataFrame(rows)
        df["fecha_reserva"] = pd.to_datetime(df["fecha_reserva"])
        df = ajustar_importes(df, objetivo_anual, pesos_mes)

        # ordenar por fecha y crear id_reserva
        df = df.sort_values("fecha_reserva").reset_index(drop=True)
        df.insert(1, "id_reserva", [f"{year}-{str(i+1).zfill(6)}" for i in range(len(df))])
        df["fecha_reserva"] = df["fecha_reserva"].dt.date
        dfs.append(df)

    clientes_df = pd.DataFrame(clientes.values())
    return dfs, clientes_df


**7. GENERAR LOS 5 AÑOS EN EXCEL**

Con un bucle, generamos cada uno de los archivos excel que necesitaremos para después analizar los resultados.

In [ ]:
dfs_generados, clientes_df = generar_todos_los_años(range(2021, 2026))

for year, df in zip(range(2021, 2026), dfs_generados):
    df.to_excel(f"reservas_flash_{year}.xlsx", index=False)

df_total = pd.concat(dfs_generados, ignore_index=True)
df_total.to_csv("reservas_flash_2021_2025.csv", index=False, encoding="utf-8-sig")

# todos los clientes (columna segmento: no usarla como variable en el análisis)
clientes_df.to_csv("clientes.csv", index=False, encoding="utf-8-sig")


**8. CARGA DE ARCHIVOS YA GENERADOS**

In [ ]:
df_reservas = pd.read_csv(r"C:\Users\Dell\Desktop\BOOTCAMP_DATA\ESPECIALIZACIÓN\S13_PROYECTO\BBDD SINTETICAS 2021-2025\reservas_flash_2021_2025.csv")

for col in ["fecha_reserva", "fecha_salida", "fecha_vuelta"]:
    df_reservas[col] = pd.to_datetime(df_reservas[col], errors='coerce')


In [21]:
df_reservas.head(5)

,fecha_reserva,id_reserva,proveedor,precio_bruto,importe,comision,descuento_real,fecha_salida,fecha_vuelta,numero_noches,...,adultos,niños,bebes,destino,pais_destino,categoria_viaje,id_cliente,edad_cliente,sexo_cliente,comunidad_autonoma_cliente
0,2021-01-01,2021-000001,Samatravel,393.98,319.12,38.29,0.19,2021-03-18,2021-03-26,8,...,1,1,0,San Sebastián,España,camping,CLT-002703,62,M,Andalucía
1,2021-01-01,2021-000002,Bambara Travel,333.84,257.06,30.85,0.23,2021-03-26,2021-03-30,4,...,3,1,0,Madrid,España,camping,CLT-002619,58,M,Andalucía
2,2021-01-01,2021-000003,Nomada,1470.58,1235.29,148.23,0.16,2021-05-16,2021-05-26,10,...,1,1,1,Lisboa,Portugal,solo hotel,CLT-002657,38,M,Castilla y León
3,2021-01-01,2021-000004,Viajes Olympia,82.94,72.16,8.66,0.13,2021-05-29,2021-05-31,2,...,2,0,1,Cáceres,España,camping,CLT-002667,61,M,Andalucía
4,2021-01-01,2021-000005,Sidertours,238.29,181.10,21.73,0.24,2021-02-21,2021-02-23,2,...,2,1,1,Benidorm,España,solo hotel,CLT-002670,18,F,Andalucía


## **✨ VALIDACIÓN DE DATOS**

Crear una serie de funciones que puedan determinar si nuestros datos sintéticos son correctos para continuar trabajando con ellos. CUestiones importante en nuestros 5 archivos ya generados:
+ ¿Es coherente?
+ ¿Es realista?
+ ¿Respeta INE y los patrones de negocio?

**1. FUNCIONES DE VALIDACIÓN**

___
**validar_estructura**
+ No hay columnas vacías
+ Tipos correctos: fechas → datetime, números → int/float
+ No hay edades < 18, noches negativas, importes 0

In [22]:
def validar_estructura(df):
    print("✔ Validación estructura y tipos")
    print(df.info())
    print(df.head())
    print(df.describe(include='all'))
    print("-"*60)

**validar_duplicados**
+ Reservas duplicadas o clientes inconsistentes.
+ Número de duplicados en id_reserva
+ Los clientes pueden repetirse, pero siempre con los mismos atributos

In [96]:
def validar_duplicados(df):
    print("✔ Validación duplicados")
    print("Duplicados en id_reserva:", df["id_reserva"].duplicated().sum())
    clientes_unicos = df[["id_cliente","edad_cliente","sexo_cliente","comunidad_autonoma_cliente"]].drop_duplicates()
    print("Clientes únicos coherentes:", clientes_unicos.shape[0])
    print("-"*60)


**validar_fechas**
+ Fechas imposibles → dataset inválido.
+ Nunca salida < reserva
+ Nunca vuelta < salida

In [97]:
def validar_fechas(df):
    print("✔ Validación fechas")
    print("Salida >= reserva:", (df["fecha_salida"] >= df["fecha_reserva"]).all())
    print("Vuelta > salida:", (df["fecha_vuelta"] > df["fecha_salida"]).all())
    print("-"*60)


**validar_noches**
+ Validamos que las noches coinciden con las fechas.

In [98]:
def validar_noches(df):
    print(" Validación noches")

    fecha_vuelta = pd.to_datetime(df["fecha_vuelta"], errors='coerce')
    fecha_salida = pd.to_datetime(df["fecha_salida"], errors='coerce')
    
    diferencia_dias = (fecha_vuelta - fecha_salida).dt.days
    
    noches_ok = (diferencia_dias == df["numero_noches"]).all()
    
    print("Noches coherentes:", noches_ok)
    print("-" * 60)


**validar_economia**
+ Importe y comisión deben ser coherentes

In [99]:
def validar_economia(df):
    print("✔ Validación económica")
    df["importe_calc"] = df["precio_bruto"] * (1 - df["descuento_real"])
    df["comision_calc"] = df["importe"] * 0.12
    print("Importe OK:", (df["importe"].round(2) == df["importe_calc"].round(2)).all())
    print("Comisión OK:", (df["comision"].round(2) == df["comision_calc"].round(2)).all())
    print("-"*60)

**validar_patrones_negocio**
+ Que las reglas impuestas de los prodcutos que más pueden vender, por ejemplo, hayan funcionado.

In [100]:
def validar_patrones_negocio(df):
    print("✔ Validación patrones de negocio")
    print("Egipto (%):", df["pais_destino"].value_counts(normalize=True).get("Egipto", 0))
    print("Japón 2024-2025:")
    print(df[df["fecha_reserva"].dt.year.isin([2024,2025])]["pais_destino"].value_counts(normalize=True))
    print("Last minute verano (días):")
    dias = (df[df["fecha_reserva"].dt.month.isin([6,7,8])]["fecha_salida"] -
            df[df["fecha_reserva"].dt.month.isin([6,7,8])]["fecha_reserva"]).dt.days.describe()
    print(dias)
    print("-"*60)


**validar_comunidades**
+ Validamos que la distribución respeta el INE.

In [101]:
def validar_comunidades(df):
    print("✔ Validación comunidades autónomas")
    print(df["comunidad_autonoma_cliente"].value_counts(normalize=True))
    print("-"*60)

**validar_outliers**
+ Comprobamos que no haya valores fuera de rango o absurdos.

In [102]:
def validar_outliers(df):
    print("✔ Validación outliers")
    print(df.describe())
    print("-"*60)

---

**2. FUNCIÓN GENERAL DE VALIDACIÓN**

In [103]:
def validar_df(df):
    print(f"\n\n================ VALIDANDO {nombre} ================\n")
    validar_estructura(df)
    validar_duplicados(df)
    validar_fechas(df)
    validar_noches(df)
    validar_economia(df)
    validar_patrones_negocio(df)
    validar_comunidades(df)
    validar_outliers(df)
    print(f"✔ Validación completa para {nombre}\n")


---

**3. VALIDAR TODOS LOS DATOS**

In [104]:
validar_df(df_reservas)




================ VALIDANDO Reservas 2025 ================

✔ Validación estructura y tipos
<class 'pandas.DataFrame'>
RangeIndex: 64119 entries, 0 to 64118
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   fecha_reserva               64119 non-null  datetime64[us]
 1   id_reserva                  64119 non-null  str           
 2   proveedor                   64119 non-null  str           
 3   precio_bruto                64119 non-null  float64       
 4   importe                     64119 non-null  float64       
 5   comision                    64119 non-null  float64       
 6   descuento_real              64119 non-null  float64       
 7   fecha_salida                64119 non-null  datetime64[us]
 8   fecha_vuelta                64119 non-null  datetime64[us]
 9   numero_noches               64119 non-null  int64         
 10  tipo_habitacion             64119 no

---

## **✨INFORME DE KPIS**

**1. FUNCIONES DE KPIs**

In [110]:
def kpi_volumen(df):
    return f"""
### ✔ Volumen
- Total reservas: {df.shape[0]}
- Total viajeros: {df['numero_viajeros_total'].sum()}
"""

def kpi_ticket(df):
    return f"""
### ✔ Ticket medio
- Importe medio: {df['importe'].mean():.2f} €
- Importe mínimo: {df['importe'].min():.2f} €
- Importe máximo: {df['importe'].max():.2f} €
"""

def kpi_margen(df):
    margen_total = df['comision'].sum()
    margen_medio = df['comision'].mean()
    return f"""
### ✔ Margen
- Margen total: {margen_total:.2f} €
- Margen medio por reserva: {margen_medio:.2f} €
"""

def kpi_anticipacion(df):
    dias = (df["fecha_salida"] - df["fecha_reserva"]).dt.days
    return f"""
### ✔ Anticipación de compra
- Media: {dias.mean():.1f} días
- Mínimo: {dias.min()} días
- Máximo: {dias.max()} días
"""

def kpi_noches(df):
    return f"""
### ✔ Duración del viaje
- Noches medias: {df['numero_noches'].mean():.1f}
- Noches mínimas: {df['numero_noches'].min()}
- Noches máximas: {df['numero_noches'].max()}
"""

def kpi_destinos(df):
    destinos = df["pais_destino"].value_counts(normalize=True).head(10)
    return f"""
### ✔ Top países destino
{destinos.to_string()}
"""

def kpi_comunidades(df):
    comunidades = df["comunidad_autonoma_cliente"].value_counts(normalize=True).head(10)
    return f"""
### ✔ Top comunidades autónomas
{comunidades.to_string()}
"""

def kpi_recurrencia(df):
    recurrencia = df["id_cliente"].value_counts()
    return f"""
### ✔ Recurrencia de clientes
- Clientes únicos: {df['id_cliente'].nunique()}
- Clientes con más de 1 reserva: {(recurrencia > 1).sum()}
- % de clientes recurrentes: {((recurrencia > 1).sum() / df['id_cliente'].nunique()):.2%}
"""

**2. GENERADOR DE DASHBOARD KPIs**

In [111]:
def generar_kpis(df):
    dashboard = f"""
# DASHBOARD KPIs — {nombre}

## 1. Volumen
{kpi_volumen(df)}

## 2. Ticket medio
{kpi_ticket(df)}

## 3. Margen
{kpi_margen(df)}

## 4. Anticipación de compra
{kpi_anticipacion(df)}

## 5. Duración del viaje
{kpi_noches(df)}

## 6. Destinos
{kpi_destinos(df)}

## 7. Comunidades autónomas
{kpi_comunidades(df)}

## 8. Recurrencia de clientes
{kpi_recurrencia(df)}

---

Dashboard generado automáticamente.
"""
    return dashboard

**3. BUCLE PARA GENERAR KPIs DE TODOS LOS AÑOS**

In [114]:
dashboard = generar_kpis(df_reservas)
print(dashboard)
print("\n" + "="*80 + "\n")



# DASHBOARD KPIs — Reservas 2025

## 1. Volumen

### ✔ Volumen
- Total reservas: 64119
- Total viajeros: 224661


## 2. Ticket medio

### ✔ Ticket medio
- Importe medio: 1255.31 €
- Importe mínimo: 13.94 €
- Importe máximo: 6576.00 €


## 3. Margen

### ✔ Margen
- Margen total: 9658680.02 €
- Margen medio por reserva: 150.64 €


## 4. Anticipación de compra

### ✔ Anticipación de compra
- Media: 66.0 días
- Mínimo: 7 días
- Máximo: 180 días


## 5. Duración del viaje

### ✔ Duración del viaje
- Noches medias: 6.2
- Noches mínimas: 1
- Noches máximas: 14


## 6. Destinos

### ✔ Top países destino
pais_destino
España       0.690232
Egipto       0.176484
Japón        0.082004
Italia       0.008110
Francia      0.004819
Noruega      0.003447
Portugal     0.003213
EEUU         0.003088
Tailandia    0.003072
Marruecos    0.003026


## 7. Comunidades autónomas

### ✔ Top comunidades autónomas
comunidad_autonoma_cliente
Andalucía               0.192049
Cataluña                0.146275
Comunit